# Train a baseline on Colab (B / C / D) and run full-scene inference

Prerequisites on Google Drive (`MyDrive/reservoir_cd/`): `chips.zip` and `reservoir_cd_code.zip`, produced locally by
`.venv/bin/python scripts/package_chips.py --with-keys --code`. Use a GPU runtime (Runtime > Change runtime type).
Checkpoints are written to Drive every epoch; if the session disconnects, rerun all cells and training resumes from `last.ckpt`.

In [ ]:
# ---- settings ---------------------------------------------------------------
CONFIG   = "configs/train_D.yaml"      # train_B.yaml | train_C.yaml | train_D.yaml
RUN_NAME = "D_prithvi_lora_r8"         # one folder per run on Drive
OVERRIDES = []                          # e.g. ["model.adaptation.r=16"], ["data.loso_holdout=Mettur"], ["data.label_source=rule"]
DRIVE = "/content/drive/MyDrive/reservoir_cd"
WORK  = "/content/work"
RUN_DIR  = f"{DRIVE}/runs/{RUN_NAME}"
PRED_DIR = f"{DRIVE}/predictions/{RUN_NAME}"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ---- code + dependencies ----------------------------------------------------
import os, subprocess
os.makedirs(WORK, exist_ok=True)
!cd {WORK} && unzip -oq {DRIVE}/reservoir_cd_code.zip
!pip install -q terratorch torchgeo lightning peft pyarrow zarr xarray rasterio pyproj geopandas scikit-image pyyaml
!cd {WORK} && pip install -q --no-deps -e .

In [ ]:
# ---- data: unzip chips from Drive to the LOCAL disk (much faster than reading from Drive) ----
!mkdir -p {WORK}/data && unzip -oq {DRIVE}/chips.zip -d {WORK}/data
!ls {WORK}/data/processed/chips | head; du -sh {WORK}/data/processed/chips; ls {WORK}/data/ground_truth 2>/dev/null

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# ---- train (resumes automatically from {RUN_DIR}/checkpoints/last.ckpt if it exists) ----
sets = [f"data.chips_dir={WORK}/data/processed/chips", f"data.ground_truth_dir={WORK}", "data.num_workers=2"] + OVERRIDES
!cd {WORK} && python -m reservoir_cd.training.train --config {CONFIG} --output-dir {RUN_DIR} --set {" ".join(sets)}

In [ ]:
# ---- training curves (CSV log on Drive) ----
import pandas as pd
m = pd.read_csv(f"{RUN_DIR}/logs/metrics.csv")
m.groupby("epoch")[[c for c in m.columns if c.startswith("val_")]].last().tail(10)

In [ ]:
# ---- full-scene inference (tiled, overlap-blended) -> Drive ----
!cd {WORK} && python -m reservoir_cd.inference.predict --config {RUN_DIR}/config.yaml \
    --ckpt {RUN_DIR}/checkpoints/last.ckpt --out {PRED_DIR} --chips-dir {WORK}/data/processed/chips --overlap 128 --batch-size 4

Outputs: `PRED_DIR/<Site>.zarr` with `pred` (P,2,H,W), `change` (P,H,W), `water_prob` (P,2,H,W). Download them (or leave on Drive) and run the evaluation locally.
To use the **best** instead of the **last** checkpoint, pass the `epoch*.ckpt` file in `RUN_DIR/checkpoints/`.